In [1]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
os.getcwd()

'd:\\work\\WBC_Segmentation\\WhileBloodCellClassification\\research'

In [3]:
os.chdir("..")

In [4]:
os.getcwd()

'd:\\work\\WBC_Segmentation\\WhileBloodCellClassification'

In [5]:
from WhiteBloodCellClassification.DomainAdaptation.DA_module import DomainAdaptationModule

DomainAdaptation = DomainAdaptationModule()

d:\work\WBC_Segmentation\WhileBloodCellClassification\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\work\WBC_Segmentation\WhileBloodCellClassification\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [7]:
class JSTCDataset(Dataset):
    def __init__(self,root_dir,mask_dir,binary_dir):
        self.mask_dir = os.path.join(root_dir, mask_dir)
        self.binary_dir = os.path.join(root_dir, binary_dir)
        self.images = sorted([f for f in os.listdir(self.mask_dir) if f.endswith('.bmp')])
        self.mask = sorted([f for f in os.listdir(self.mask_dir) if f.endswith('.png')])
        self.binary = sorted([f for f in os.listdir(self.binary_dir) if f.endswith('.png')])
    
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = self.mask[idx]
        binary_name = self.binary[idx]
        img_path = os.path.join(self.mask_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        binary_path = os.path.join(self.binary_dir, binary_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        image = transform(image)
        mask = cv2.imread(mask_path,0)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        label_mask = np.zeros_like(mask)
        label_mask[mask == 255] = 2
        label_mask[mask == 128] = 1
        label_mask[mask == 0] = 0
        label_mask = torch.tensor(label_mask).long()
        binary = cv2.imread(binary_path,0)
        binary = cv2.resize(binary, (256, 256), interpolation=cv2.INTER_NEAREST)
        label_binary = np.zeros_like(binary)
        label_binary[binary == 255] = 1
        label_binary[binary == 0] = 0
        label_binary = torch.tensor(label_binary).long()
        return {
            "image": image,
            "mask": label_mask,
            "binary": label_binary
        }

In [8]:
root_dir = r"D:\work\WBC_Segmentation\WhileBloodCellClassification\data/RawData"
mask_dir = r"Dataset 1"
binary_dir = r"Result 1"

In [9]:
dataset = JSTCDataset(root_dir, mask_dir, binary_dir)
dataloader = DataLoader(dataset, batch_size=10, shuffle=False)
for batch in dataloader:
    images = batch["image"]
    masks = batch["mask"]
    binarys = batch["binary"]
    print(images.shape, masks.shape, binarys.shape)
    break

torch.Size([10, 3, 256, 256]) torch.Size([10, 256, 256]) torch.Size([10, 256, 256])


In [10]:
len(dataloader)

30

In [11]:
image = images[0]
print(image.shape)

torch.Size([3, 256, 256])


In [12]:
mask = masks[2]
mask.shape

torch.Size([256, 256])

In [13]:
binary = binarys[2]
binary.shape

torch.Size([256, 256])

In [14]:
LEARNING_RATE = 2e-4
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 1
NUM_WORKERS = 4
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256
PIN_MEMORY = True
LOAD_MODEL = False
weight_cross_entropy = 2
weight_mask = 5
weight_rec = 1


In [15]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])

def check_accuracy(loader,model,device="cuda"):
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    model.eval()
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            y = y.to(device)
            preds = torch.sigmoid(model(x))
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)
    print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}")
    print(f"Dice score: {dice_score/len(loader)}")
    model.train()
    
def get_optimizer(model, learning_rate=0.0002, weight_decay=0.01):
    """
    Khởi tạo optimizer cho domain adaptation training
    
    Args:
        model: Model cần train
        learning_rate: Learning rate (theo bài báo: 0.0002)
        weight_decay: Weight decay để tránh overfitting
    """
    # AdamW được khuyến nghị cho transformer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=(0.9, 0.999)
    )
    return optimizer

def get_scaler():
    """
    Khởi tạo gradient scaler cho mixed precision training
    """
    scaler = torch.cuda.amp.GradScaler()
    return scaler

In [16]:
from tqdm import tqdm

In [17]:
loop = tqdm(dataloader)
for batch_idx in loop:
        images = batch["image"]
        masks = batch["mask"]
        binarys = batch["binary"]
        images = images.to(device=DEVICE)
        masks = masks.float().unsqueeze(1).to(device=DEVICE)
        binarys = binarys.float().unsqueeze(1).to(device=DEVICE)

print(images.shape, masks.shape, binarys.shape)

100%|██████████| 30/30 [00:01<00:00, 16.63it/s]

torch.Size([10, 3, 256, 256]) torch.Size([10, 1, 256, 256]) torch.Size([10, 1, 256, 256])


In [18]:
def training_fn(loader, model, optimizer, CrossEntropyLoss,DICELoss,BCELoss, ReconstructionLoss, scaler):
    loop = tqdm(loader)
    
    for batch_idx in loop:
        images = batch["image"]
        masks = batch["mask"]
        binarys = batch["binary"]
        images = images.to(device=DEVICE)
        masks = masks.float().to(device=DEVICE)
        masks_long = masks.long().to(device=DEVICE)
        binarys = binarys.float().to(device=DEVICE)

        # forward
        with torch.cuda.amp.autocast():
            predictions, encoded_features, query_features = model(images)
            CrossEntropy = CrossEntropyLoss(predictions, masks_long)
            mask_loss = 0
            K = predictions.shape[1]
            for k in range(K):
                pred_binary  = predictions[:, k:k+1, :, :]
                target_binary = (masks == k).float()
                dice = DICELoss(pred_binary, target_binary.long())
                bce = BCELoss(pred_binary, target_binary.unsqueeze(1))
                mask_loss += bce + dice
            Reconstruction = ReconstructionLoss(encoded_features, query_features)
            loss = weight_cross_entropy * CrossEntropy + weight_mask * mask_loss + weight_rec * Reconstruction

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop        
        loop.set_postfix(loss=loss.item())


In [19]:
os.getcwd()

'd:\\work\\WBC_Segmentation\\WhileBloodCellClassification'

In [20]:
from WhiteBloodCellClassification.DomainAdaptation.DA_module import DomainAdaptationModule
from WhiteBloodCellClassification.models.losses import DiceLoss, BCELoss, ReconstructionLoss, CLSLoss

In [21]:
model = DomainAdaptationModule().to(DEVICE)
CrossEntropyLoss = CLSLoss()
diceLoss = DiceLoss()
bceLoss = BCELoss()
reconstructionLoss = ReconstructionLoss()
optimizer = get_optimizer(model)
scaler = get_scaler()
training_fn(loader=dataloader, model=model, optimizer=optimizer, CrossEntropyLoss=CrossEntropyLoss, DICELoss=diceLoss, BCELoss=bceLoss, ReconstructionLoss=reconstructionLoss, scaler=scaler)

d:\work\WBC_Segmentation\WhileBloodCellClassification\.venv\Lib\site-packages\torch\cuda\amp\grad_scaler.py:125: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
  0%|          | 0/30 [00:00<?, ?it/s]d:\work\WBC_Segmentation\WhileBloodCellClassification\.venv\Lib\site-packages\torch\amp\autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|██████████| 30/30 [20:32<00:00, 41.10s/it, loss=17.1]


In [ ]:
model = DomainAdaptationModule().to(DEVICE)
for name, param in model.named_parameters():
    print(name, param.shape)